In [ ]:
%%capture
!pip install datasets

### Imports

In [ ]:
import torch
import torch.nn as nn

from keras.preprocessing.sequence import pad_sequences
import numpy as np

import torch.optim as optim
import time
import random, math

from tqdm import tqdm

# Transformer

### Embedding Layer (Word + Positional)

Instead of using a sinusoidal positional encoding (as in the original paper), a learnable positional embedding is used.

In [ ]:
# auxiliary functions
def base_3_conversion(number):
    # Funzione per la conversione di un numero in base 10 in base 3
    quotient, remainder = divmod(number,3)
    result = [remainder]
    while quotient > 0:
        quotient, remainder = divmod(quotient,3)
        result.append(remainder)  # Inserisci il resto all'inizio della lista
    return result

def base_3_list(n):
    # Funzione per ottenere una lista di liste dei numeri da 0 a 9 convertiti in base 3
    result = []
    for num in range(n):
        converted_num = base_3_conversion(num)
        result.append(converted_num)
    return result


In [ ]:
class EmbeddingTrained(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, dropout=0.1):
    super(EmbeddingTrained, self).__init__()
    self.word_embed = nn.Embedding(vocab_size, embed_dim)
    self.pos_embed = nn.Embedding(max_length, embed_dim)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    batch_size, seq_length = x.shape
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    positions = torch.arange(0, seq_length).expand(
        batch_size, seq_length).to(device)
    embedding = self.word_embed(x) + self.pos_embed(positions)
    return self.dropout(embedding)

In [ ]:
class EmbeddingBase3Pos(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, dropout=0.1):
    super(EmbeddingBase3Pos, self).__init__()

    log_len = math.ceil(math.log(max_length) / math.log(3))

    self.word_embed = nn.Embedding(vocab_size, embed_dim - log_len)

    base_3_representation = base_3_list(max_length)
    self.pos_embed = torch.tensor(pad_sequences(base_3_representation, maxlen=log_len, truncating="post", padding="post", dtype="int") - 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    batch_size, seq_length = x.shape
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    aux = self.pos_embed.unsqueeze(0).repeat(x.size(0), 1, 1).to(device)
    embedding = torch.cat((self.word_embed(x), aux), dim = 2)
    return self.dropout(embedding)

class EmbeddingBase3PosExpand(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, reduced_embed = 16, dropout=0.1):
    super(EmbeddingBase3PosExpand, self).__init__()

    log_len = math.ceil(math.log(max_length) / math.log(3))
    self.word_embed = nn.Embedding(vocab_size, reduced_embed)
    self.expand_layer = nn.Linear(reduced_embed, embed_dim - log_len)
    # self.word_embed = nn.Embedding(vocab_size, embed_dim - log_len)

    base_3_representation = base_3_list(max_length)
    self.pos_embed = torch.tensor(pad_sequences(base_3_representation, maxlen=log_len, truncating="post", padding="post", dtype="int") - 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    batch_size, seq_length = x.shape
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    word_embeddings = self.expand_layer(self.word_embed(x))
    pos_embeddings = self.pos_embed.unsqueeze(0).repeat(x.size(0), 1, 1).to(device)
    embedding = torch.cat((word_embeddings, pos_embeddings), dim = 2)
    return self.dropout(embedding)

### Multi-Head Self-Attention


In [ ]:
class MHSelfAttention(nn.Module):
  def __init__(self, embed_dim, num_heads):
    super(MHSelfAttention, self).__init__()
    self.embed_dim = embed_dim
    self.num_heads = num_heads
    self.head_dim = embed_dim // num_heads

    assert (self.num_heads*self.head_dim == self.embed_dim),'embed size must be divisible by number of heads'

    self.w_queries = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
    self.w_keys = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
    self.w_values = nn.Linear(self.embed_dim, self.embed_dim, bias=False)

    self.fc_out = nn.Linear(self.head_dim*self.num_heads , self.embed_dim)

  def forward(self, x):

    # shape of x = [batch_size, sentence_length, embedding_dim]
    batch_size = x.shape[0]
    sentence_len = x.shape[1]

    queries = self.w_queries(x).reshape(
        batch_size, sentence_len, self.num_heads, self.head_dim).permute(
            0, 2, 1, 3)

    keys = self.w_keys(x).reshape(
        batch_size, sentence_len, self.num_heads, self.head_dim).permute(
            0, 2, 3, 1)


    values = self.w_values(x).reshape(
        batch_size, sentence_len, self.num_heads, self.head_dim).permute(
            0, 2, 1, 3)

    attention_scores = torch.einsum('bijk,bikl->bijl', queries, keys)
    attention_dist = torch.softmax(attention_scores /
                               (self.embed_dim ** (1/2)), dim=-1)
    attention_out = torch.einsum('bijk,bikl->bijl', attention_dist, values)
    concatenated_out = attention_out.permute(0, 2, 1, 3).reshape(
        batch_size, sentence_len, self.embed_dim)

    return concatenated_out

In [ ]:
class EfficientAttention(nn.Module):
  def __init__(self, embed_dim, num_heads):
    super(EfficientAttention, self).__init__()
    self.embed_dim = embed_dim

    self.w_queries = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
    self.w_output = nn.Linear(self.embed_dim, self.embed_dim, bias=False)


  def forward(self, x):

    # shape of x = [batch_size, sentence_length, embedding_dim]
    batch_size = x.shape[0]
    sentence_len = x.shape[1]

    queries = self.w_queries(x) #shape [batch_size, sentence_length, embedding_dim]

    attention_scores = torch.einsum('bij,bjk->bik', queries, torch.transpose(x,1,2))

    attention_dist = torch.softmax(attention_scores / (self.embed_dim ** (1/2)), dim=-1)

    attention_out = torch.einsum('bij,bjk->bik', attention_dist, x)

    out = self.w_output(attention_out)

    return out

### Transformer Encoder


In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self, embed_dim, num_heads, forward_expansion, dropout=0.1):
    super(TransformerEncoder, self).__init__()

    self.attention = MHSelfAttention(embed_dim, num_heads)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.feed_forward = nn.Sequential(
        nn.Linear(embed_dim, int(forward_expansion*embed_dim)),
        nn.GELU(),
        nn.Linear(int(forward_expansion*embed_dim), embed_dim)
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    attention_out = self.dropout(self.attention(x))
    x = self.norm1(x + attention_out)
    forward_out = self.dropout(self.feed_forward(x))
    out = self.norm2(x + forward_out)

    return out

In [ ]:
class EfficientTransformerEncoder(nn.Module):
  def __init__(self, embed_dim, num_heads, forward_expansion, dropout=0.1):
    super(EfficientTransformerEncoder, self).__init__()

    self.attention = EfficientAttention(embed_dim, num_heads)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.feed_forward = nn.Sequential(
        nn.Linear(embed_dim, int(forward_expansion*embed_dim)),
        nn.GELU(),
        nn.Linear(int(forward_expansion*embed_dim), embed_dim)
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    attention_out = self.dropout(self.attention(x))
    x = self.norm1(x + attention_out)
    forward_out = self.dropout(self.feed_forward(x))
    out = self.norm2(x + forward_out)

    return out

In [ ]:
class EfficientConvTransformerEncoder(nn.Module):
  def __init__(self, embed_dim, num_heads, forward_expansion, dropout=0.1):
    super(EfficientConvTransformerEncoder, self).__init__()

    self.attention = EfficientAttention(embed_dim, num_heads)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.feed_forward = nn.Sequential(
        nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim),
        nn.GELU(),
        nn.Conv1d(embed_dim, embed_dim, kernel_size=1)
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):

    attention_out = self.dropout(self.attention(x))
    x = self.norm1(x + attention_out)
    forward_out = self.dropout(self.feed_forward(x.transpose(1, 2))).transpose(1, 2)
    out = self.norm2(x + forward_out)
    return out

In [ ]:
class EfficientSWIGLUTransformerEncoder(nn.Module):
  def __init__(self, embed_dim, num_heads, forward_expansion, dropout=0.1):
    super(EfficientSWIGLUTransformerEncoder, self).__init__()

    self.attention = EfficientAttention(embed_dim, num_heads)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.fc_up1 = nn.Linear(embed_dim, int(forward_expansion*embed_dim))
    self.fc_up2 = nn.Linear(embed_dim, int(forward_expansion*embed_dim))
    self.fc_down = nn.Linear(int(forward_expansion*embed_dim), embed_dim)
    self.silu = torch.nn.SiLU(inplace=False)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    attention_out = self.dropout(self.attention(x))
    x = self.norm1(x + attention_out)
    mid1 = self.fc_up1(x)
    mid2 = self.silu(self.fc_up2(x))
    mid = torch.mul(mid1,mid2)
    swiglu = self.fc_down(mid)
    forward_out = self.dropout(swiglu)
    out = self.norm2(x + forward_out)

    return out

### End-to-End Classifier

1. An embedding layer
2. A single transformer encoder layer
3. A fully-connected network as a linear classifier

In [ ]:
class ClassifierV1(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, num_heads, forward_expansion):
      super(ClassifierV1, self).__init__()

      self.embedder = EmbeddingTrained(vocab_size, max_length, embed_dim)
      self.encoder = TransformerEncoder(embed_dim, num_heads, forward_expansion)
      self.fc = nn.Linear(embed_dim, N_LABELS)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.sigmoid(out)

In [ ]:
class ClassifierV2(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, num_heads, forward_expansion):
      super(ClassifierV2, self).__init__()

      self.embedder = EmbeddingTrained(vocab_size, max_length, embed_dim)
      self.encoder = EfficientTransformerEncoder(embed_dim, num_heads, forward_expansion)
      self.fc = nn.Linear(embed_dim, N_LABELS)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.softmax(out, 1)

In [ ]:
class ClassifierV3(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, num_heads, forward_expansion, layers):
      super(ClassifierV3, self).__init__()

      self.embedder = EmbeddingTrained(vocab_size, max_length, embed_dim)
      blocks = []
      blocks += [EfficientConvTransformerEncoder(embed_dim, num_heads, forward_expansion) for _ in range(layers)]
      self.encoder = nn.Sequential(*blocks)
      self.fc = nn.Linear(embed_dim, N_LABELS)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.softmax(out, 1)

In [ ]:
class ClassifierV4(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, num_heads, forward_expansion, layers):
      super(ClassifierV4, self).__init__()

      self.embedder = EmbeddingBase3Pos(vocab_size, max_length, embed_dim)
      blocks = []
      blocks += [EfficientTransformerEncoder(embed_dim, num_heads, forward_expansion) for _ in range(layers)]
      self.encoder = nn.Sequential(*blocks)
      self.fc = nn.Linear(embed_dim, N_LABELS)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.softmax(out, 1)

In [ ]:
class ClassifierV7(nn.Module):
  def __init__(self, vocab_size, max_length, red_embed_dim, embed_dim, num_heads, forward_expansion, layers):
      super(ClassifierV7, self).__init__()

      self.embedder = EmbeddingBase3PosExpand(vocab_size, max_length, embed_dim, red_embed_dim)
      blocks = []
      blocks += [EfficientSWIGLUTransformerEncoder(embed_dim, num_heads, forward_expansion) for _ in range(layers)]
      self.encoder = nn.Sequential(*blocks)
      self.fc = nn.Linear(embed_dim, N_LABELS)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.sigmoid(out)

In [ ]:
# Print the model size
def print_model_size(model):
  param_size = 0
  param_count = 0
  for param in model.parameters():
    param_size += param.nelement() * param.element_size()
    param_count += param.nelement()
  buffer_size = 0
  for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

  size_all_mb = (param_size + buffer_size) / 1024**2
  print('Model params: {:.3f}M'.format(param_count/1e6))
  print('Model size: {:.3f}MB'.format(size_all_mb))

# Load and Preprocess Datasets

In [ ]:
VOCAB_SIZE = 512*8
MAX_LENGTH = 512

In [ ]:
from datasets import load_dataset
import sentencepiece as spm
import os

# https://huggingface.co/datasets/tweet_eval
# https://huggingface.co/datasets/super_glue

#'Amazon', "imdb", "sst2" "sst5" "twitter"
DATASET = "sst2"


#load dataset
if DATASET == "imdb":
  dataset = load_dataset("imdb")
  train_data = dataset['train']
  test_data = dataset['test']

  text_train = train_data.to_dict()["text"]
  label_train = train_data.to_dict()["label"]

  text_test = test_data.to_dict()["text"]
  label_test = test_data.to_dict()["label"]

  N_LABELS = len(set(label_train))

elif DATASET == "Amazon": #NEEDS FIXING: LABELS ARE 0 through 4 but only 0 and 4 are used...
  dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", trust_remote_code=True)

  text_full = [tit + " " + txt for tit, txt in zip(dataset["full"]["title"], dataset["full"]["text"])]
  label_full = dataset["full"]["rating"]
  label_full = (torch.Tensor(label_full) - 1).tolist() #rescaling labels to 0-4

  text_train = text_full[:len(label_full)//10 * 9]
  label_train = label_full[:len(label_full)//10 * 9]

  text_test = text_full[len(label_full)//10 * 9 + 1: ]
  label_test = label_full[len(label_full)//10 * 9 + 1: ]

  N_LABELS = len(set(label_full))

elif DATASET == "sst2":
  dataset = load_dataset("sst2")

  train_data = dataset['train']
  val_data = dataset['validation']
  # test_data = dataset['test'] #sti idioti non hanno classificato il test set

  text_train = train_data.to_dict()["sentence"]
  label_train = train_data.to_dict()["label"]

  text_test = val_data.to_dict()["sentence"]
  label_test = val_data.to_dict()["label"]

  N_LABELS = len(set(label_train))

elif DATASET == "sst5":
  dataset = load_dataset("SetFit/sst5")

  train_data = dataset['train']
  val_data = dataset['validation']
  test_data = dataset['test']

  text_train = train_data.to_dict()["text"]
  label_train = train_data.to_dict()["label"]

  text_test = test_data.to_dict()["text"] +  val_data.to_dict()["text"]
  label_test = test_data.to_dict()["label"] + val_data.to_dict()["label"]

  N_LABELS = len(set(label_train))

elif DATASET == "twitter":
  dataset = load_dataset("tweet_eval", "emoji")
  train_data = dataset['train']
  val_data = dataset['validation']
  test_data = dataset['test']

  text_train = train_data["text"] + val_data["text"]
  label_train = train_data["label"] + val_data["label"]

  text_test = test_data["text"]
  label_test = test_data["label"]

  N_LABELS = len(set(label_train))


assert len(text_train) == len(label_train)
assert len(text_test) == len(label_test)
assert(N_LABELS > 1)

if not os.path.exists("./train_ds_" + DATASET + ".txt"):
  #reduce dataset for sentencepiece training
  try:
    idxs = random.sample(range(len(text_train)), 10000)
    aux = [text_train[i] for i in idxs]
  except:
    aux = text_train
  #save file of dataset for tokenizer
  filename = "./train_ds_" + DATASET + ".txt"
  with open(filename, 'w') as f:
    for s in aux:
      f.write(s)

if not os.path.exists("./m_" + DATASET + ".model"):
  #Train tokenizer
  spm.SentencePieceTrainer.train(input="./train_ds_" + DATASET + ".txt", model_prefix="m_" + DATASET, max_sentence_length = 100000000 ,vocab_size=VOCAB_SIZE)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
#Load tokenizer
sp = spm.SentencePieceProcessor(model_file="m_" + DATASET + ".model")

print(f'Dictionary size {sp.get_piece_size()}')
print(f'Vocabulary: {[sp.id_to_piece(id) for id in range(sp.get_piece_size())]}')
print(f'Encoding results:  {sp.encode("this is a phrase that could be commonly found", out_type=str)} -> {sp.encode("this is a phrase that could be commonly found")}')

Dictionary size 4096
Vocabulary: ['<unk>', '<s>', '</s>', '▁', '▁#', 's', '▁@', '...', '.', ',', 'user', "'", '▁the', '!', 't', '▁️', '▁to', '▁a', 'ing', '▁I', '▁in', 'y', '▁my', 'e', '▁you', '▁with', '▁of', '▁and', '▁for', 'm', 'd', 'I', 'a', '▁this', 'n', 'in', '▁is', 'r', 'S', '▁California', 'er', 'ed', 'o', 'A', 'l', '▁at', 'i', 'c', 're', '#', 'u', '▁on', '▁it', 'p', 'b', 'amp', 'w', '▁The', '▁love', '▁S', 'al', ';', '▁me', 'T', '▁&', ')', 'k', 'C', 'h', '▁be', '-', '▁so', 'B', 'la', 'E', 'on', '▁San', '▁B', 'at', '▁Los', '@', 'le', 'g', 'an', '▁A', '▁day', '▁from', '▁Angeles', 'the', 'th', 'The', '!!', 'D', 'ro', '▁was', '▁-', '▁C', 'P', ':', '...#', 'f', 'O', '_', '▁by', 'L', '▁Beach', 'v', 'ly', 'F', '▁your', 'es', '▁our', 'H', 'it', 'M', 'or', '▁:', 'My', '▁that', '▁(', 'z', '▁all', '▁night', '▁P', '▁time', '"', 'W', '▁are', 'ri', 'love', '▁out', 'R', 'to', '▁up', 'ar', '▁today', 'st', '▁T', 'is', 'x', 'G', '▁like', '/', 'Happy', '▁w', 'en', 'li', '▁Vegas', '▁we', 'ra', '!!!',

In [ ]:
train_tokens = list(map(lambda t: [1] + sp.encode(t)[:MAX_LENGTH - 2] + [2], text_train))
test_tokens = list(map(lambda t: [1] + sp.encode(t)[:MAX_LENGTH - 2] + [2], text_test))

In [ ]:
train_tokens_ids = pad_sequences(train_tokens, maxlen = MAX_LENGTH, truncating="post", padding="post", dtype="int")
test_tokens_ids = pad_sequences(test_tokens, maxlen = MAX_LENGTH, truncating="post", padding="post", dtype="int")

In [ ]:
train_masks = train_tokens_ids > 0
test_masks = test_tokens_ids > 0

In [ ]:
train_tokens_tensor = torch.tensor(train_tokens_ids)
#train_y_tensor = torch.tensor(np.array(label_train).reshape(-1, 1)).long()
train_y_tensor = torch.tensor(np.array(label_train)).long()

test_tokens_tensor = torch.tensor(test_tokens_ids)
#test_y_tensor = torch.tensor(np.array(label_test).reshape(-1, 1)).long()
test_y_tensor = torch.tensor(np.array(label_test)).long()

train_masks_tensor = torch.tensor(train_masks).float()
test_masks_tensor = torch.tensor(test_masks).float()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

train_dataset = TensorDataset(train_tokens_tensor, train_masks_tensor, train_y_tensor)
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=32)

test_dataset = TensorDataset(test_tokens_tensor, test_masks_tensor, test_y_tensor)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size=32)

# Model creation

In [ ]:
EMBED_DIM = 128
REDUCED_EMBEDDING_DIM = 16
NUM_HEADS = 8
FORWARD_EXPANSION = 0.5
LAYERS = 1


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#initialize model
classifier_1 = ClassifierV1(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION)
classifier_1.to(device)

#print all model parameters with names
print("--------------MultiHead original classifier--------------")
for name, param in classifier_1.named_parameters():
  print(f"{name}: {param.nelement()}")
#print the model size
print_model_size(classifier_1)

classifier_2 = ClassifierV2(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION)
classifier_2.to(device)

#print all model parameters with names
print("--------------Efficient Attention classifier--------------")
for name, param in classifier_2.named_parameters():
  print(f"{name}: {param.nelement()}")
#print the model size
print_model_size(classifier_2)


classifier_3 = ClassifierV3(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION, LAYERS)
classifier_3.to(device)

#print all model parameters with names
print("--------------Efficient Attention classifier LAYERED --------------")
for name, param in classifier_3.named_parameters():
  print(f"{name}: {param.nelement()}")
#print the model size
print_model_size(classifier_3)

classifier_4 = ClassifierV4(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION, LAYERS)
classifier_4.to(device)

#print all model parameters with names
print("--------------Efficient Attention classifier no learned pos --------------")
for name, param in classifier_4.named_parameters():
  print(f"{name}: {param.nelement()}")
#print the model size
print_model_size(classifier_4)

classifier_7 = ClassifierV7(VOCAB_SIZE, MAX_LENGTH, REDUCED_EMBEDDING_DIM, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION, LAYERS)
classifier_7.to(device)

#print all model parameters with names
print("--------------Efficient Attention added SWIGLU and encoder reduction --------------")
for name, param in classifier_7.named_parameters():
  print(f"{name}: {param.nelement()}")
#print the model size
print_model_size(classifier_7)

--------------MultiHead original classifier--------------
embedder.word_embed.weight: 524288
embedder.pos_embed.weight: 65536
encoder.attention.w_queries.weight: 16384
encoder.attention.w_keys.weight: 16384
encoder.attention.w_values.weight: 16384
encoder.attention.fc_out.weight: 16384
encoder.attention.fc_out.bias: 128
encoder.norm1.weight: 128
encoder.norm1.bias: 128
encoder.norm2.weight: 128
encoder.norm2.bias: 128
encoder.feed_forward.0.weight: 8192
encoder.feed_forward.0.bias: 64
encoder.feed_forward.2.weight: 8192
encoder.feed_forward.2.bias: 128
fc.weight: 2560
fc.bias: 20
Model params: 0.675M
Model size: 2.576MB
--------------Efficient Attention classifier--------------
embedder.word_embed.weight: 524288
embedder.pos_embed.weight: 65536
encoder.attention.w_queries.weight: 16384
encoder.attention.w_output.weight: 16384
encoder.norm1.weight: 128
encoder.norm1.bias: 128
encoder.norm2.weight: 128
encoder.norm2.bias: 128
encoder.feed_forward.0.weight: 8192
encoder.feed_forward.0.bia

# Training

In [ ]:
EPOCHS = 10
LR = 1e-4

In [ ]:
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion.to(device);

In [ ]:
from tqdm import tqdm

def trainer(model, lr):

  optimizer = optim.Adam(model.parameters(), lr=lr)

  for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    tqdm_train_loader = tqdm(train_dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for step_num, batch_data in enumerate(tqdm_train_loader):

        token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

        logits = model(token_ids)

        batch_loss = criterion(logits, labels)
        train_loss += batch_loss.item()

        model.zero_grad()
        batch_loss.backward()


        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        log_step = 50
        if step_num % log_step == (log_step - 1):
          tqdm_train_loader.set_postfix(loss = train_loss / log_step)
          train_loss = 0

## Training classifiers

In [ ]:
trainer(classifier_1, LR)

KeyboardInterrupt: 

In [ ]:
trainer(classifier_2, LR)

In [ ]:
trainer(classifier_3, LR)

In [ ]:
trainer(classifier_4, LR)

In [ ]:
trainer(classifier_7, LR)

Epoch 1:  77%|███████▋  | 1208/1563 [11:55<03:20,  1.77it/s, loss=2.79]

# Evaluation

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

def evaluator(model):
  model.eval()
  predicted = torch.Tensor([])
  all_logits = torch.Tensor([])

  tqdm_test_loader = tqdm(test_dataloader, desc=f"Evaluation: ", leave=False)

  with torch.no_grad():
      for step_num, batch_data in enumerate(tqdm_test_loader):

          token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

          logits = model(token_ids)
          loss = criterion(logits, labels)
          numpy_logits = logits.cpu().detach()
          predicted = torch.cat((predicted, torch.argmax(numpy_logits, dim = 1)))
          all_logits  = torch.cat((all_logits, numpy_logits))
  print()

  return predicted.tolist();

In [ ]:
predicted = evaluator(classifier_1)

print(classification_report(label_test, predicted))

conf_matrix = confusion_matrix(label_test, predicted)

# Print or visualize the confusion matrix
# print("Confusion Matrix:")
# print(conf_matrix)

In [ ]:
predicted = evaluator(classifier_2)

print(classification_report(label_test, predicted))

conf_matrix = confusion_matrix(label_test, predicted)

# Print or visualize the confusion matrix
# print("Confusion Matrix:")
# print(conf_matrix)


In [ ]:
predicted = evaluator(classifier_3)

print(classification_report(label_test, predicted))

conf_matrix = confusion_matrix(label_test, predicted)

# Print or visualize the confusion matrix
# print("Confusion Matrix:")
# print(conf_matrix)


In [ ]:
predicted = evaluator(classifier_4)

print(classification_report(label_test, predicted))

conf_matrix = confusion_matrix(label_test, predicted)

# Print or visualize the confusion matrix
# print("Confusion Matrix:")
# print(conf_matrix)


In [ ]:
predicted = evaluator(classifier_7)

print(classification_report(label_test, predicted))

conf_matrix = confusion_matrix(label_test, predicted)

# Print or visualize the confusion matrix
# print("Confusion Matrix:")
# print(conf_matrix)


# Averaging runs

In [ ]:
N_RUNS = 5

In [ ]:
predictions = []

for i in range(N_RUNS):
  print(f"Running iteration {i}")

  classifier = ClassifierV7(VOCAB_SIZE, MAX_LENGTH, REDUCED_EMBEDDING_DIM, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION, LAYERS)
  classifier.to(device)

  trainer(classifier, LR)

  predictions += evaluator(classifier)



In [ ]:
print(classification_report(label_test*N_RUNS, predictions))

# AUX tests

In [ ]:
# custom_text = "I really liked this film, it was for sure worth watching. The color correction was sublime, It really gave a lot to the film"
custom_text = "This film sucked, I hated it, it was a waste of time. All actors were terrible and even the lights were terrible"
enc = [1] + sp.encode(custom_text)[:510] + [2]
enc = enc + [0]* (512-len(enc))
enc_tensor = torch.tensor(enc).unsqueeze(0).to(device)
classifier(enc_tensor)